# LangGraph 009 — Iterative (Looping) Workflows

A generate → evaluate → optimize loop, built from one edge pointing backwards.
Needs `langgraph`; **no API key** — the three models are scripted.

| Part | What we check |
|---|---|
| A | the state: keys that replace against keys that accumulate |
| B | 3 nodes, 6 runs — the first workflow whose cost depends on the data |
| C | where the iteration counter starts changes the bill |
| D | no cap: `GraphRecursionError` at **10,007** super-steps, 5,003 calls in |
| E | the cap's exit hands back a draft nobody approved |
| F | no path map, no loop — and no error either |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Two kinds of key in one state

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

class TweetState(TypedDict):
    topic: str
    tweet: str                                        # replaced every round
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str                                     # replaced every round
    iteration: int                                    # the loop counter
    max_iteration: int                                # the cap
    tweet_history: Annotated[list[str], operator.add]     # appended
    feedback_history: Annotated[list[str], operator.add]  # appended

# Two kinds of key in one state. `tweet` and `feedback` hold the current round;
# the two history keys accumulate every round, because a reducer is attached.
# `iteration` and `max_iteration` are the only reason this workflow is
# guaranteed to stop.

In [ ]:
# Three scripted "models" so the loop runs with no API key. The critic turns
# down the first two drafts and then approves - stand in for how strict a real
# evaluator model happens to be.
FEEDBACK = ["Reads like a joke book. Try an observation, not a punchline.",
            "Better, but the second clause explains the first. Cut it."]
calls = {"generate": 0, "evaluate": 0, "optimize": 0}

def generate_tweet(state: TweetState) -> dict:
    calls["generate"] += 1
    draft = f"draft 1 on {state['topic']}"
    return {"tweet": draft, "tweet_history": [draft]}

def evaluate_tweet(state: TweetState) -> dict:
    calls["evaluate"] += 1
    if calls["evaluate"] > len(FEEDBACK):
        return {"evaluation": "approved", "feedback": "Ship it.",
                "feedback_history": ["Ship it."]}
    note = FEEDBACK[calls["evaluate"] - 1]
    return {"evaluation": "needs_improvement", "feedback": note,
            "feedback_history": [note]}

def optimize_tweet(state: TweetState) -> dict:
    calls["optimize"] += 1
    iteration = state["iteration"] + 1          # the counter moves HERE
    draft = f"draft {iteration} on {state['topic']}"
    return {"tweet": draft, "iteration": iteration, "tweet_history": [draft]}

# The router. Two exits: approved, or out of iterations.
def route_evaluation(state: TweetState) -> Literal["approved", "needs_improvement"]:
    if state["evaluation"] == "approved":
        return "approved"
    if state["iteration"] >= state["max_iteration"]:
        return "approved"                       # here "approved" means "stop"
    return "needs_improvement"

## Part B — The back-edge, and what it costs

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TweetState)
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_evaluation,
                            {"approved": END, "needs_improvement": "optimize"})
graph.add_edge("optimize", "evaluate")      # <-- this back-edge IS the loop

workflow = graph.compile()
result = workflow.invoke({"topic": "Indian Railways", "tweet": "", "feedback": "",
                          "evaluation": "needs_improvement", "iteration": 1,
                          "max_iteration": 5,
                          "tweet_history": [], "feedback_history": []})

print(calls)
print("drafts:", len(result["tweet_history"]), "| final:", result["tweet"])
print("verdict:", result["evaluation"])

# {'generate': 1, 'evaluate': 3, 'optimize': 2}
# drafts: 3 | final: draft 3 on Indian Railways
# verdict: approved
#
# Three nodes ran six times. A loop is not new machinery - it is one edge
# pointing backwards, and the first workflow in this track whose number of
# node runs depends on the data.

In [ ]:
assert calls == {"generate": 1, "evaluate": 3, "optimize": 2}
assert len(result["tweet_history"]) == 3          # one draft per round
assert len(result["feedback_history"]) == 3
assert result["tweet"] == "draft 3 on Indian Railways"   # `tweet` REPLACES
assert result["feedback_history"][-1] == "Ship it."
print("3 nodes ran 6 times; the history kept every round, `tweet` kept one")

# a kinder critic costs less, and the graph is identical
FEEDBACK.clear()                                  # approves the first draft
calls.update(generate=0, evaluate=0, optimize=0)
quick = workflow.invoke({"topic": "monsoon", "tweet": "", "feedback": "",
                         "evaluation": "needs_improvement", "iteration": 1,
                         "max_iteration": 5, "tweet_history": [],
                         "feedback_history": []})
print(calls, "->", len(quick["tweet_history"]), "draft")
assert calls["optimize"] == 0 and len(quick["tweet_history"]) == 1

## Part C — What `max_iteration` actually bounds

A critic that never approves, so only the cap can stop the loop. Guess how many
rewrites a cap of 5 buys before running it.

In [ ]:
from langgraph.graph import StateGraph, START, END

def never_happy(state: TweetState) -> dict:
    calls["evaluate"] += 1
    return {"evaluation": "needs_improvement", "feedback": "No.",
            "feedback_history": ["No."]}

def build_capped():
    graph = StateGraph(TweetState)
    graph.add_node("generate", generate_tweet)
    graph.add_node("evaluate", never_happy)
    graph.add_node("optimize", optimize_tweet)
    graph.add_edge(START, "generate")
    graph.add_edge("generate", "evaluate")
    graph.add_conditional_edges("evaluate", route_evaluation,
                                {"approved": END, "needs_improvement": "optimize"})
    graph.add_edge("optimize", "evaluate")
    return graph.compile()

for start_at in (1, 0):
    calls.update(generate=0, evaluate=0, optimize=0)
    out = build_capped().invoke({"topic": "trains", "tweet": "", "feedback": "",
                                 "evaluation": "needs_improvement",
                                 "iteration": start_at, "max_iteration": 5,
                                 "tweet_history": [], "feedback_history": []})
    print(f"iteration starts at {start_at}: optimize ran {calls['optimize']}x, "
          f"{len(out['tweet_history'])} drafts")
    if start_at == 1:
        assert calls["optimize"] == 4
    else:
        assert calls["optimize"] == 5
print("the cap bounds the COUNTER; the calls are the distance it travels")

## Part D — And if you forget the cap?

The widely quoted recursion limit is 25. Find out what it is here.

In [ ]:
# What stops a loop whose critic never approves? Take the cap out and see.
from langgraph.errors import GraphRecursionError

def never_happy(state: TweetState) -> dict:
    calls["evaluate"] += 1
    return {"evaluation": "needs_improvement", "feedback": "No.",
            "feedback_history": ["No."]}

def route_without_cap(state: TweetState) -> Literal["approved", "needs_improvement"]:
    return "approved" if state["evaluation"] == "approved" else "needs_improvement"

graph = StateGraph(TweetState)
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", never_happy)
graph.add_node("optimize", optimize_tweet)
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_without_cap,
                            {"approved": END, "needs_improvement": "optimize"})
graph.add_edge("optimize", "evaluate")

calls.update(generate=0, evaluate=0, optimize=0)
blank = {"topic": "trains", "tweet": "", "feedback": "",
         "evaluation": "needs_improvement", "iteration": 1,
         "max_iteration": 10**9, "tweet_history": [], "feedback_history": []}
try:
    graph.compile().invoke(blank)
except GraphRecursionError as e:
    print(str(e).splitlines()[0][:70])
print("critic calls:", calls["evaluate"], "| optimizer calls:", calls["optimize"])

# Recursion limit of 10007 reached without hitting a stop condition
# critic calls: 5003 | optimizer calls: 5003
#
# There is a backstop, and on langgraph 1.1.6 it is 10,007 super-steps - not
# the 25 that is often quoted. Five thousand model calls is not a safety net
# anybody wants to rely on. Set `recursion_limit` yourself, and count
# iterations in the state, which is what max_iteration is for.

In [ ]:
# bound it yourself, and the work is bounded in proportion
for limit in (10, 25, 50):
    calls.update(generate=0, evaluate=0, optimize=0)
    try:
        graph.compile().invoke(blank, config={"recursion_limit": limit})
    except GraphRecursionError:
        pass
    print(f"recursion_limit {limit:>3} -> evaluate {calls['evaluate']:>2}, "
          f"optimize {calls['optimize']:>2}")
    assert calls["evaluate"] <= limit
print()
print("A crash guard, not a budget: it stops a runaway expensively and with no")
print("reason attached. The state counter is what stops a loop on purpose.")

## Part E — The quiet exit

In [ ]:
calls.update(generate=0, evaluate=0, optimize=0)
capped = build_capped().invoke({"topic": "trains", "tweet": "", "feedback": "",
                                "evaluation": "needs_improvement", "iteration": 1,
                                "max_iteration": 3, "tweet_history": [],
                                "feedback_history": []})
print("evaluation field:", repr(capped["evaluation"]))
print("tweet returned  :", repr(capped["tweet"]))
assert capped["evaluation"] == "needs_improvement"     # never approved
print()
print("The run ended normally. Only the state says the draft was rejected.")

# the fix: give the cap its own ending
def route_with_flag(state: TweetState):
    if state["evaluation"] == "approved":
        return "approved"
    if state["iteration"] >= state["max_iteration"]:
        return "gave_up"
    return "needs_improvement"

def gave_up(state: TweetState) -> dict:
    return {"feedback_history": ["STOPPED: out of iterations, draft unapproved"]}

graph2 = StateGraph(TweetState)
graph2.add_node("generate", generate_tweet)
graph2.add_node("evaluate", never_happy)
graph2.add_node("optimize", optimize_tweet)
graph2.add_node("gave_up", gave_up)
graph2.add_edge(START, "generate")
graph2.add_edge("generate", "evaluate")
graph2.add_conditional_edges("evaluate", route_with_flag,
                             {"approved": END, "needs_improvement": "optimize",
                              "gave_up": "gave_up"})
graph2.add_edge("optimize", "evaluate")
graph2.add_edge("gave_up", END)
calls.update(generate=0, evaluate=0, optimize=0)
flagged = graph2.compile().invoke({"topic": "trains", "tweet": "", "feedback": "",
                                   "evaluation": "needs_improvement", "iteration": 1,
                                   "max_iteration": 3, "tweet_history": [],
                                   "feedback_history": []})
print(flagged["feedback_history"][-1])
assert flagged["feedback_history"][-1].startswith("STOPPED")

## Part F — Who checks the labels, and when

`"approved"` is not a node name, so the path map is what gives it meaning.
Three ways to get it wrong, and they do not fail the same way.

In [ ]:
# (a) the router as written - annotated Literal - with the path map removed
graph3 = StateGraph(TweetState)
graph3.add_node("generate", generate_tweet)
graph3.add_node("evaluate", never_happy)
graph3.add_node("optimize", optimize_tweet)
graph3.add_edge(START, "generate")
graph3.add_edge("generate", "evaluate")
graph3.add_conditional_edges("evaluate", route_evaluation)      # no path map
graph3.add_edge("optimize", "evaluate")
try:
    graph3.compile()
    print("(a) compiled")
except ValueError as e:
    print("(a) compile() ->", e)

# (b) the same router annotated `-> str` instead
def route_as_str(state: TweetState) -> str:
    return route_evaluation(state)

graph4 = StateGraph(TweetState)
graph4.add_node("generate", generate_tweet)
graph4.add_node("evaluate", never_happy)
graph4.add_node("optimize", optimize_tweet)
graph4.add_edge(START, "generate")
graph4.add_edge("generate", "evaluate")
graph4.add_conditional_edges("evaluate", route_as_str)          # no path map
graph4.add_edge("optimize", "evaluate")
calls.update(generate=0, evaluate=0, optimize=0)
out = graph4.compile().invoke({"topic": "trains", "tweet": "", "feedback": "",
                               "evaluation": "needs_improvement", "iteration": 1,
                               "max_iteration": 5, "tweet_history": [],
                               "feedback_history": []})
print("(b) compiled;", calls, "drafts:", len(out["tweet_history"]))
assert calls["optimize"] == 0 and len(out["tweet_history"]) == 1
print()
print("The Literal becomes an implicit path map and compile() checks its")
print("targets. Annotated `-> str`, the same mistake is a silent no-op at run")
print("time - no loop, no error. In a loop that looks exactly like success.")

## What this notebook established

- A loop is one edge pointing backwards. Three nodes ran **six** times when the
  critic rejected twice, and twice when it approved at once.
- `tweet` replaces each round; `tweet_history` with `operator.add` accumulates —
  the history is the audit trail.
- `max_iteration` bounds the **counter**, so a cap of 5 bought 4 rewrites from
  `iteration=1` and 5 from `iteration=0`.
- With no cap, `GraphRecursionError` fired at **10,007** super-steps after
  **5,003** evaluator calls — not the 25 usually quoted.
- The cap's exit returns a draft nobody approved, unless you give it its own
  terminal node.
- Without the path map, an annotated router is a **compile-time
  `ValueError`** naming the unknown target; the same router typed `-> str`
  compiles and the loop silently never forms.

## Exercises

1. Make the evaluator approve on a coin flip and run the graph 50 times. Plot
   the distribution of rewrites.
2. Add a `stopped_early: bool` to the state instead of a separate node. Which
   version would you rather debug?
3. Put the counter in the evaluator instead of the optimizer. How does the bill
   change, and why?
4. Give the loop a second exit condition — stop if the feedback repeats — and
   measure how often it fires.